# Playground · gensim 词向量

练习用，**不是 A1 提交**。

- `most_similar_words(word)`：最近邻
- `analogy(a, b, c)`：解读为 **a : b :: c : ?**  
  例：`analogy("man", "king", "woman")` ≈ `"queen"`  
  运算：`vec(b) - vec(a) + vec(c)`

加载顺序：本地 `data/glove.6B.50d.txt` → gensim downloader → **toy 向量**（离线可玩类比）。

In [ ]:
from pathlib import Path
import pprint
import numpy as np
from gensim.models import KeyedVectors

DATA_DIR = Path("data")
LOCAL_GLOVE = DATA_DIR / "glove.6B.50d.txt"


def load_local_glove(path=LOCAL_GLOVE):
    path = Path(path)
    if not path.exists():
        return None
    print(f"Loading local GloVe: {path}")
    # GloVe 文件无 header
    return KeyedVectors.load_word2vec_format(path, binary=False, no_header=True)


def load_gensim_glove(name="glove-wiki-gigaword-50"):
    import gensim.downloader as api
    print(f"Downloading/loading via gensim: {name}")
    return api.load(name)


def build_toy_vectors(dim=50, seed=0):
    """构造一个能演示 man:king::woman:queen 的小词表（仅 playground）。"""
    rng = np.random.default_rng(seed)
    gender = rng.normal(size=dim)
    royal = rng.normal(size=dim)
    city = rng.normal(size=dim)
    country = rng.normal(size=dim)
    base = rng.normal(size=dim)

    vectors = {
        "man": base,
        "woman": base + gender,
        "king": base + royal,
        "queen": base + royal + gender,  # == king - man + woman
        "france": country,
        "paris": country + city,
        "italy": country + rng.normal(size=dim) * 0.1 + rng.normal(size=dim),
        "rome": None,  # fill below
        "movie": rng.normal(size=dim),
        "film": None,
        "actor": None,
        "good": rng.normal(size=dim),
        "better": None,
        "bad": None,
        "worse": None,
        "stanford": rng.normal(size=dim),
    }
    # italy/rome: italy : rome :: france : paris  => rome = paris - france + italy
    italy = country + rng.normal(size=dim)
    vectors["italy"] = italy
    vectors["rome"] = vectors["paris"] - vectors["france"] + italy

    film_shift = rng.normal(size=dim) * 0.2
    vectors["film"] = vectors["movie"] + film_shift
    vectors["actor"] = vectors["movie"] + rng.normal(size=dim) * 0.5

    grade = rng.normal(size=dim)
    vectors["better"] = vectors["good"] + grade
    vectors["bad"] = vectors["good"] + rng.normal(size=dim)
    vectors["worse"] = vectors["bad"] + grade

    words = list(vectors.keys())
    mat = np.stack([vectors[w] for w in words]).astype(np.float32)
    # 归一化，让 cosine 更稳
    mat /= np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9

    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(words, mat)
    print(f"Using TOY vectors | vocab={len(words)} dim={dim} (offline demo)")
    return kv


def load_word_vectors():
    wv = load_local_glove()
    if wv is not None:
        return wv, "local-glove"
    try:
        return load_gensim_glove(), "gensim-glove-50"
    except Exception as e:
        print("gensim download failed:", type(e).__name__, e)
        return build_toy_vectors(), "toy"


wv, SOURCE = load_word_vectors()
print("source:", SOURCE, "| vocab:", len(wv.index_to_key), "| dim:", wv.vector_size)

## 工具函数

In [ ]:
def _norm(w: str) -> str:
    return w.strip().lower()


def most_similar_words(word, topn=10, model=None):
    """返回与 word 最相似的 topn 个 (word, score)。"""
    model = model or wv
    word = _norm(word)
    if word not in model.key_to_index:
        raise KeyError(f"OOV: {word!r} not in vocab")
    return model.most_similar(word, topn=topn)


def similarity(w1, w2, model=None):
    """两词余弦相似度。"""
    model = model or wv
    return float(model.similarity(_norm(w1), _norm(w2)))


def analogy(a, b, c, topn=5, model=None):
    """a : b :: c : ?

    例: analogy('man', 'king', 'woman') ≈ 'queen'
    运算: vec(b) - vec(a) + vec(c)
    """
    model = model or wv
    a, b, c = _norm(a), _norm(b), _norm(c)
    for w in (a, b, c):
        if w not in model.key_to_index:
            raise KeyError(f"OOV: {w!r} not in vocab")

    neighbors = model.most_similar(positive=[b, c], negative=[a], topn=topn + 3)
    banned = {a, b, c}
    filtered = [(w, float(s)) for w, s in neighbors if w not in banned]
    return filtered[:topn]


def analogy_best(a, b, c, model=None):
    """只返回最可能的那个词。"""
    return analogy(a, b, c, topn=1, model=model)[0][0]


print("ready: most_similar_words / similarity / analogy / analogy_best")

## Demo · 词比较

In [ ]:
print("most_similar_words('king'):")
pprint.pprint(most_similar_words("king", topn=5))

print("\nmost_similar_words('movie'):")
try:
    pprint.pprint(most_similar_words("movie", topn=5))
except KeyError as e:
    print(e)

print("\nsimilarity:")
for pair in [("king", "queen"), ("king", "man"), ("movie", "film")]:
    try:
        print(f"  {pair[0]:8s} ~ {pair[1]:8s} = {similarity(*pair):.3f}")
    except KeyError as e:
        print(" ", e)

## Demo · 类比

`analogy(a, b, c)` ⇒ **a 之于 b，如同 c 之于 ?**

In [ ]:
demos = [
    ("man", "king", "woman"),     # -> queen
    ("france", "paris", "italy"), # -> rome
]

for a, b, c in demos:
    best = analogy_best(a, b, c)
    print(f"{a} : {b} :: {c} : ?  =>  {best}")
    pprint.pprint(analogy(a, b, c, topn=5))
    print()

assert analogy_best("man", "king", "woman") == "queen"
print("assert OK: analogy('man','king','woman') == 'queen'")

## 自己玩

若要用真正的 GloVe：把 `glove.6B.50d.txt` 放到本目录下的 `data/`，再重跑第 1 个 code cell。

In [ ]:
print(analogy_best("man", "king", "woman"))
pprint.pprint(most_similar_words("queen", topn=5))